In [ ]:
import pandas as pd
import csv

data = pd.read_csv("../data/all_data.csv")
data["wikiprojects"] = data["wikiprojects"].str.split("|")

data = data.dropna()

for _, row in data.iterrows():
	assert type(row["articlename"]) == str
	assert type(row["lead"]) == str
	assert type(row["wikiprojects"]) == list

keep_projects = []
with open("../data/wikiprojects.csv", "r") as f:
	reader = csv.reader(f)
	for line in reader:
		keep_projects.append(line[0])

for idx, row in data.iterrows():
	new_projects = [wp for wp in row["wikiprojects"] if wp in keep_projects]
	data.at[idx, "wikiprojects"] = new_projects

data

,articlename,lead,wikiprojects
0,Antinous,"Antinous, also called Antinoös, (; Ancient Gre...",[WikiProject Biography]
1,Anthony Parker,"Anthony Michael Parker (born June 19, 1975) is...","[WikiProject Biography, WikiProject United Sta..."
2,Armando Calderón Sol,Armando Calderón Sol (24 June 1948 – 9 October...,[WikiProject Biography]
3,Anna Akhmatova,Anna Andreyevna Gorenko (23 June [O.S. 11 June...,"[WikiProject Biography, WikiProject Russia]"
4,Arjen Robben,Arjen Robben (Dutch pronunciation: [ˈɑrjə(n) ˈ...,"[WikiProject Biography, WikiProject Football]"
...,...,...,...
51045,Johanna Maislinger,Johanna Maislinger (born 23 October 1985) is a...,"[WikiProject Biography, WikiProject Aviation]"
51046,Jenifer Bamuturaki,Jenifer Bamuturaki is a Ugandan businesswoman ...,"[WikiProject Biography, WikiProject Africa, Wi..."
51047,John Barnett (whistleblower),"John Mitchell Barnett (February 23, 1962 – Mar...","[WikiProject Biography, WikiProject Aviation]"
51048,John C. Burkhart,John Conner Burkhart (1880–1926) was an aviati...,[WikiProject Biography]


In [43]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data["wikiprojects"])

assert all([wp in keep_projects for wp in mlb.classes_])
mlb.classes_

array(['WikiProject Africa', 'WikiProject Albums',
       'WikiProject Australia', 'WikiProject Aviation',
       'WikiProject Biography', 'WikiProject Canada',
       'WikiProject Christianity', 'WikiProject Cities',
       'WikiProject Film', 'WikiProject Football', 'WikiProject France',
       'WikiProject Geography', 'WikiProject Germany',
       'WikiProject India', 'WikiProject Iran', 'WikiProject Japan',
       'WikiProject Military history', 'WikiProject Olympics',
       'WikiProject Plants', 'WikiProject Poland', 'WikiProject Russia',
       'WikiProject Songs', 'WikiProject Television',
       'WikiProject Trains', 'WikiProject United States'], dtype=object)

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_text, X_test_text, y_train, y_test = train_test_split(
	data["lead"], y, test_size=0.2, random_state=12
)

vectorizer = TfidfVectorizer(
	max_features=10000,
	stop_words="english",
	ngram_range=(1, 2),
	min_df=5
)

X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

In [ ]:
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, Dropout # type: ignore

model = Sequential()

model.add(Dense(512, input_dim=X_train.shape[1], activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(256, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(y.shape[1], activation="sigmoid"))

model.compile(
	optimizer="adam",
	loss="binary_crossentropy",
	metrics=["accuracy"]
)

model.summary()
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.1)

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_15 (Dense)            (None, 512)               5120512   
                                                                 
 dropout_10 (Dropout)        (None, 512)               0         
                                                                 
 dense_16 (Dense)            (None, 256)               131328    
                                                                 
 dropout_11 (Dropout)        (None, 256)               0         
                                                                 
 dense_17 (Dense)            (None, 25)                6425      
                                                                 
Total params: 5258265 (20.06 MB)
Trainable params: 5258265 (20.06 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


2026-04-08 21:47:11.069614: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1467880000 exceeds 10% of free system memory.


Epoch 1/5
1147/1147 [==============================] - 36s 31ms/step - loss: 0.1041 - accuracy: 0.6240 - val_loss: 0.0517 - val_accuracy: 0.7097
Epoch 2/5
1147/1147 [==============================] - 35s 31ms/step - loss: 0.0402 - accuracy: 0.7316 - val_loss: 0.0515 - val_accuracy: 0.7038
Epoch 3/5
1147/1147 [==============================] - 34s 29ms/step - loss: 0.0256 - accuracy: 0.7500 - val_loss: 0.0559 - val_accuracy: 0.7121
Epoch 4/5
1147/1147 [==============================] - 34s 29ms/step - loss: 0.0171 - accuracy: 0.7575 - val_loss: 0.0628 - val_accuracy: 0.7141
Epoch 5/5
1147/1147 [==============================] - 35s 31ms/step - loss: 0.0120 - accuracy: 0.7581 - val_loss: 0.0740 - val_accuracy: 0.7008


In [ ]:
import joblib

model.save("../model/default/wiki_model.keras")
joblib.dump(vectorizer, "../model/default/tfidf_vectorizer.pkl")
joblib.dump(mlb, "../model/default/mlb_binarizer.pkl")

['mlb_binarizer.pkl']